# 14 · 세장 기둥 — KDS 14 20 20 4.4

| 항목 | 식 | 조문 |
|---|---|---|
| 회전반지름 | $r = 0.3h$ (직사각형) | 4.4.1 |
| 세장비 한계 | $34 - 12(M_1/M_2) \le 40$ / 22 | 4.4.1 |
| 휨강성 | $EI = 0.4E_cI_g/(1+\beta_{dns})$ | 4.4 |
| 임계좌굴하중 | $P_c = \pi^2EI/(kl_u)^2$ | 4.4 |
| 모멘트확대계수 | $\delta_{ns} = C_m/(1-P_u/0.75P_c) \ge 1.0$ | 4.4.2 |
| 최소 편심 모멘트 | $M_{2,min} = P_u(15+0.03h)$ | 4.4.2 |

In [1]:
import matplotlib.pyplot as plt
import numpy as np

# 한글 글꼴이 없는 환경에서도 그림이 깨지지 않도록 축 라벨은 ASCII 로 둔다
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 96

In [2]:
from concreteproperties import ConcreteSection
from sectionproperties.pre.library import concrete_rectangular_section

from concreteproperties_kds import KDS


def beam_section(fck=27, fy=400):
    """400 x 600 보 단면 (상부 2-D16, 하부 4-D22, 피복 50 mm)."""
    kds = KDS(column_type="tie")
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=600, b=400,
        dia_top=16, area_top=198.6, n_top=2, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=4, c_bot=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec


def column_section(fck=27, fy=400, column_type="tie"):
    """500 x 500 기둥 단면 (8-D22, 피복 50 mm)."""
    kds = KDS(column_type=column_type)
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=500, b=500,
        dia_top=22, area_top=387.1, n_top=3, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=3, c_bot=50,
        dia_side=22, area_side=387.1, n_side=1, c_side=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec

In [3]:
from concreteproperties_kds.slender import check_slenderness

P_U, M1, M2, H = 1500e3, 90e6, 150e6, 500.0

kds, conc_sec = column_section()
conc = conc_sec.concrete_geometries[0].material
gross = kds.get_transformed_gross_properties(
    elastic_modulus=conc.elastic_modulus
)

for l_u in [3000.0, 6000.0, 9000.0]:
    res = check_slenderness(
        p_u=P_U, m1=M1, m2=M2, k=1.0, l_u=l_u, h=H,
        e_c=conc.elastic_modulus, i_g=gross.ixx_c,
        braced=True, beta_dns=0.6,
    )
    print(f"### lu = {l_u:.0f} mm")
    res.print_results()

    f_res, _, phi = kds.ultimate_bending_capacity(n_design=P_U)
    ratio = res.m_c / f_res.m_x
    print(f"설계 휨강도 phi*Mn = {f_res.m_x / 1e6:.2f} kN.m (phi = {phi:.3f})")
    print(f"소요/강도          = {ratio:.3f}"
          f"  {'만족' if ratio <= 1.0 else '불만족'}")
    print()

### lu = 3000 mm
세장 기둥 검토 (KDS 14 20 20 4.4)
회전반지름           r      =       150.00 mm
세장비           k*lu/r     =        20.00
한계 세장비                 =        26.80
세장 기둥                   =          아니오
----------------------------------------------------------------
세장효과를 무시할 수 있습니다 (Mc = M2).


설계 휨강도 phi*Mn = 357.17 kN.m (phi = 0.702)
소요/강도          = 0.420  만족

### lu = 6000 mm
세장 기둥 검토 (KDS 14 20 20 4.4)
회전반지름           r      =       150.00 mm
세장비           k*lu/r     =        40.00
한계 세장비                 =        26.80
세장 기둥                   =            예
----------------------------------------------------------------
휨강성               EI     =   3.8366e+13 N.mm^2
임계좌굴하중         Pc     =     10518.38 kN
                 0.75Pc     =      7888.79 kN
                     Cm     =       0.8400
모멘트확대계수   delta_ns   =       1.0372
----------------------------------------------------------------
단부 모멘트          M2     =       150.00 kN.m
최소 편심 모멘트     M2,min =        45.00 kN.m
설계 모멘트          Mc     =       155.58 kN.m


설계 휨강도 phi*Mn = 357.17 kN.m (phi = 0.702)
소요/강도          = 0.436  만족

### lu = 9000 mm
세장 기둥 검토 (KDS 14 20 20 4.4)
회전반지름           r      =       150.00 mm
세장비           k*lu/r     =        60.00
한계 세장비                 =        26.80
세장 기둥                   =            예
----------------------------------------------------------------
휨강성               EI     =   3.8366e+13 N.mm^2
임계좌굴하중         Pc     =      4674.84 kN
                 0.75Pc     =      3506.13 kN
                     Cm     =       0.8400
모멘트확대계수   delta_ns   =       1.4681
----------------------------------------------------------------
단부 모멘트          M2     =       150.00 kN.m
최소 편심 모멘트     M2,min =        45.00 kN.m
설계 모멘트          Mc     =       220.21 kN.m


설계 휨강도 phi*Mn = 357.17 kN.m (phi = 0.702)
소요/강도          = 0.617  만족



## 비지지 길이에 따른 모멘트 확대

In [4]:
l_u = np.linspace(2000, 11000, 200)
delta, m_c, slender = [], [], []

for length in l_u:
    r = check_slenderness(
        p_u=P_U, m1=M1, m2=M2, k=1.0, l_u=float(length), h=H,
        e_c=conc.elastic_modulus, i_g=gross.ixx_c,
    )
    delta.append(r.delta_ns)
    m_c.append(r.m_c / 1e6)
    slender.append(r.slender)

f_res, _, _ = kds.ultimate_bending_capacity(n_design=P_U)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(l_u, delta)
axes[0].axhline(1.0, ls=":", color="grey", lw=0.8)
axes[0].set_xlabel("unsupported length, lu (mm)")
axes[0].set_ylabel("delta_ns")
axes[0].set_title("Moment magnifier")
axes[0].grid(alpha=0.3)

axes[1].plot(l_u, m_c, label="Mc")
axes[1].axhline(
    f_res.m_x / 1e6, ls="--", color="tab:red",
    label=f"phi*Mn = {f_res.m_x / 1e6:.0f}",
)
axes[1].axhline(M2 / 1e6, ls=":", color="grey", lw=0.8, label="M2")
axes[1].set_xlabel("unsupported length, lu (mm)")
axes[1].set_ylabel("design moment (kN.m)")
axes[1].set_title("Magnified design moment")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)
fig.tight_layout()

first_slender = l_u[np.argmax(slender)]
print(f"세장 기둥이 되는 비지지 길이 ~ {first_slender:.0f} mm")

세장 기둥이 되는 비지지 길이 ~ 4035 mm


비지지 길이가 길어질수록 $\delta_{ns}$ 가 급격히 커진다. $P_u$ 가
$0.75P_c$ 에 접근하면 분모가 0 에 가까워지기 때문이다. 그 지점을 넘으면
좌굴이므로 `check_slenderness` 가 `ValueError` 를 낸다.